In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import gymnasium as gym
import gym
import gym_pygame
import imageio

from collections import deque
from torch.distributions import Categorical
from huggingface_hub import notebook_login

%matplotlib inline

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [75]:
env_id = "CartPole-v1"

env = gym.make(env_id)
eval_env = gym.make(env_id, render_mode = "rgb_array")

s_size = env.observation_space.shape[0]
a_size = int(env.action_space.n)

rand_s = env.observation_space.sample
rand_a = env.action_space.sample

s_size, a_size, rand_s, rand_a

(4,
 2,
 <bound method Box.sample of Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)>,
 <bound method Discrete.sample of Discrete(2)>)

In [ ]:
class Policy(nn.Module):

    def __init__(self, s_size, a_size, h_size):
        super(Policy, self).__init__()

        self.Layer = nn.Sequential(
            nn.Linear(
                s_size,
                h_size
            ),
            nn.ReLU(),
            nn.Linear(
                h_size,
                h_size*2
            ),
            nn.ReLU(),
            nn.Linear(
                h_size*2,
                a_size
            )
        ) 

    def forward(self, x):

        x = self.Layer(x)

        return F.softmax(x, dim = 1)

    def act(self, state):
        
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)
        probs = self.forward(state).cpu()
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action)


In [24]:
def Reinforce(policy, optimizer, n_training_episodes, max_t, gamma, print_every):

    scores_deque = deque(maxlen = 100)
    scores = []

    for i_episode in range(1, n_training_episodes+1):
        saved_log_probs = []
        rewards = []

        state = env.reset()
        done = False

        for t in range(max_t):
            
            action, log_probs = policy.act(state)
            saved_log_probs.append(log_probs)

            state, reward, done, _ = env.step(action)
            #done = terminated or truncated
            rewards.append(reward)

            if done:
                break
        
        scores_deque.append(sum(rewards))
        scores.append(sum(rewards))

        returns = deque(maxlen = max_t)
        n_steps = len(rewards)

        for t in range(n_steps)[::-1]:

            disc_return_t = (returns[0] if len(returns)>0 else 0)
            returns.appendleft(gamma * disc_return_t + rewards[t])

        eps = np.finfo(np.float32).eps.item()

        returns = torch.tensor(returns)
        returns = (returns - returns.mean()) / (returns.std() + eps)

        policy_loss = []

        for log_prob, disc_return in zip(saved_log_probs, returns):
            policy_loss.append(-log_prob * disc_return)

        policy_loss = torch.cat(policy_loss).sum()

        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        if i_episode % print_every == 0:

            print("Episode {}\tAverage Score: {:.2f}".format(i_episode, np.mean(scores_deque)))

    return scores

In [112]:
cartpole_hyperparameters = {
    "h_size": 16,
    "n_training_episodes": 1000,
    "n_evaluation_episodes": 10,
    "max_t": 1000,
    "gamma": 1.0,
    "lr": 1e-2,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [113]:
cartpole_policy = Policy(
    cartpole_hyperparameters["state_space"],
    cartpole_hyperparameters["action_space"],
    cartpole_hyperparameters["h_size"]
).to(device)

cartpole_optimizer = optim.Adam(cartpole_policy.parameters(), lr = cartpole_hyperparameters["lr"])

In [116]:
scores = Reinforce(
    cartpole_policy,
    cartpole_optimizer,
    cartpole_hyperparameters["n_training_episodes"],
    cartpole_hyperparameters["max_t"],
    cartpole_hyperparameters["gamma"],
    100,
)

Episode 100	Average Score: 62.50
Episode 200	Average Score: 309.04
Episode 300	Average Score: 399.20
Episode 400	Average Score: 369.39
Episode 500	Average Score: 478.99
Episode 600	Average Score: 494.70
Episode 700	Average Score: 498.66
Episode 800	Average Score: 491.79
Episode 900	Average Score: 494.99
Episode 1000	Average Score: 478.43


In [117]:
def evaluate_agent(env, max_steps, n_eval_episodes, policy):

    episode_rewards = []

    for ep in range(n_eval_episodes):

        state, _ = env.reset()
        step = 0
        done = False
        total_rewards_ep = 0

        for step in range(max_steps):
            action, _ = policy.act(state)
            new_state, reward, truncated, terminated, info = env.step(action)
            total_rewards_ep += reward
            done = terminated or truncated

            if done:
                break

            state = new_state
        
        episode_rewards.append(total_rewards_ep)
    
    mean_rewards = np.mean(episode_rewards)
    std_rewards = np.std(episode_rewards)

    return mean_rewards, std_rewards

In [118]:
evaluate_agent(eval_env, cartpole_hyperparameters["max_t"], cartpole_hyperparameters["n_evaluation_episodes"], cartpole_policy)

(np.float64(500.0), np.float64(0.0))

In [ ]:
from safetensors.torch import save_file

# Save only the weight dictionary safely
save_file(cartpole_policy.state_dict(), path / "model.safetensors")

notebook_login()

In [132]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json
import imageio

import tempfile

import os

In [136]:
def record_video(env, policy, out_directory, fps=30):
    """
    Generate a replay video of the agent
    :param env
    :param Qtable: Qtable of our agent
    :param out_directory
    :param fps: how many frame per seconds (with taxi-v3 and frozenlake-v1 we use 1)
    """
    images = []
    done = False
    state, _ = env.reset()
    img = env.render()
    images.append(img)
    while not done:
        # Take the action (index) that have the maximum expected future reward given that state
        action, _ = policy.act(state)
        state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated  # We directly put next_state = state for recording logic
        img = env.render()
        images.append(img)
    imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [140]:
def push_to_hub(repo_id,
                model,
                hyperparameters,
                eval_env,
                video_fps=30
                ):
  """
  Evaluate, Generate a video and Upload a model to Hugging Face Hub.
  This method does the complete pipeline:
  - It evaluates the model
  - It generates the model card
  - It generates a replay video of the agent
  - It pushes everything to the Hub

  :param repo_id: repo_id: id of the model repository from the Hugging Face Hub
  :param model: the pytorch model we want to save
  :param hyperparameters: training hyperparameters
  :param eval_env: evaluation environment
  :param video_fps: how many frame per seconds to record our video replay
  """

  _, repo_name = repo_id.split("/")
  api = HfApi()

  # Step 1: Create the repo
  repo_url = api.create_repo(
        repo_id=repo_id,
        exist_ok=True,
  )

  with tempfile.TemporaryDirectory() as tmpdirname:
    local_directory = Path(tmpdirname)

    # Step 2: Save the model
    #torch.save(model, local_directory / "model.pt")

    # Step 3: Save the hyperparameters to JSON
    with open(local_directory / "hyperparameters.json", "w") as outfile:
      json.dump(hyperparameters, outfile)

    # Step 4: Evaluate the model and build JSON
    mean_reward, std_reward = evaluate_agent(eval_env,
                                            hyperparameters["max_t"],
                                            hyperparameters["n_evaluation_episodes"],
                                            model)
    # Get datetime
    eval_datetime = datetime.datetime.now()
    eval_form_datetime = eval_datetime.isoformat()

    evaluate_data = {
          "env_id": hyperparameters["env_id"],
          "mean_reward": mean_reward,
          "n_evaluation_episodes": hyperparameters["n_evaluation_episodes"],
          "eval_datetime": eval_form_datetime,
    }

    # Write a JSON file
    with open(local_directory / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = hyperparameters["env_id"]

    metadata = {}
    metadata["tags"] = [
          env_name,
          "reinforce",
          "reinforcement-learning",
          "custom-implementation",
          "deep-rl-class"
      ]

    # Add metrics
    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
      )

    # Merges both dictionaries
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Reinforce** Agent playing **{env_id}**
  This is a trained model of a **Reinforce** agent playing **{env_id}** .
  To learn to use this model and train yours check Unit 4 of the Deep Reinforcement Learning Course: https://huggingface.co/deep-rl-course/unit4/introduction
  """

    readme_path = local_directory / "README.md"
    readme = ""
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
          readme = f.read()
    else:
      readme = model_card

    with readme_path.open("w", encoding="utf-8") as f:
      f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)

    # Step 6: Record a video

    # Step 7. Push everything to the Hub
    api.upload_folder(
          repo_id=repo_id,
          folder_path=local_directory,
          path_in_repo=".",
    )

    print(f"Your model is pushed to the Hub. You can view your model here: {repo_url}")

In [ ]:
repo_id = # "your repo id"
push_to_hub(
    repo_id,
    cartpole_policy,  # The model we want to save
    cartpole_hyperparameters,  # Hyperparameters
    eval_env,  # Evaluation environment
)

In [26]:
import numpy as np

# Dynamically patch NumPy 2.x to support legacy Gym's check
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

In [20]:
env_id = "Pixelcopter-PLE-v0"

env = gym.make(env_id)
eval_env = gym.make(env_id)

s_size = env.observation_space.shape[0]
a_size = int(env.action_space.n)


In [21]:
pixelcopter_hyperparameters = {
    "h_size": 64,
    "n_training_episodes": 50000,
    "n_evaluation_episodes": 10,
    "max_t": 10000,
    "gamma": 0.99,
    "lr": 1e-4,
    "env_id": env_id,
    "state_space": s_size,
    "action_space": a_size,
}

In [22]:
pixelcopter_policy = Policy(
    pixelcopter_hyperparameters["state_space"],
    pixelcopter_hyperparameters["action_space"],
    pixelcopter_hyperparameters["h_size"]
).to(device)

pixelcopter_optimizer = optim.Adam(pixelcopter_policy.parameters(), lr = pixelcopter_hyperparameters["lr"])

In [ ]:
scores = Reinforce(
    policy = pixelcopter_policy,
    optimizer = pixelcopter_optimizer,
    gamma = pixelcopter_hyperparameters["gamma"],
    n_training_episodes = pixelcopter_hyperparameters["n_training_episodes"],
    max_t = pixelcopter_hyperparameters["max_t"],
    print_every = 1e3
)

# same as above after training push to hub